In [1]:
import io
import zipfile
import requests
import frontmatter


In [2]:
url = 'https://codeload.github.com/debauchee/barrier-wiki/faq/zip/refs/heads/main'
resp = requests.get(url)

In [3]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/master'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [4]:
barrier_docs = read_repo_data('debauchee', 'barrier-wiki')

In [5]:
print(f"barrier docs: {len(barrier_docs)}")

barrier docs: 10


In [6]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [7]:
barrier_chunks = []

for doc in barrier_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    barrier_chunks.extend(chunks)

In [8]:
print(type(barrier_chunks[0]))
print(barrier_chunks[0].keys())

<class 'dict'>
dict_keys(['start', 'chunk', 'filename'])


In [9]:
from pprint import pprint
pprint(barrier_chunks[0]['chunk'][:200])

('## Adding Barrier to the Windows Firewall\n'
 '\n'
 'Screenshots captured by [l0o0](https://github.com/l0o0)\n'
 '\n'
 '![search windows '
 'defender](https://user-images.githubusercontent.com/2743744/49057950-ca40bd00-f23c-')


In [10]:
import re
text = barrier_docs[6]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

In [11]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.

    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    #For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)

    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1] # "##" + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)

    return sections
    

In [12]:
barrier_chunks = []

for doc in barrier_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        barrier_chunks.append(section_doc)

In [24]:
from minsearch import Index

barrier_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

barrier_index.fit(barrier_chunks)

In [14]:
print(barrier_chunks[0].keys())

dict_keys(['filename', 'section'])


In [15]:
barrier_faq = read_repo_data('debauchee', 'barrier-wiki')



In [16]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [19]:
record = de_barrier_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)

NameError: name 'de_barrier_faq' is not defined

In [17]:
query = 'How do I install barrier on Mac OS?'
v_query = embedding_model.encode(query)

In [18]:
query = 'How do I install barrier on Mac OS?'
results = index.search(query)

In [17]:
similarity = v_query.dot(v_doc)

NameError: name 'v_doc' is not defined

In [20]:
from tqdm.auto import tqdm
from minsearch import VectorSearch
import numpy as np

barrier_embeddings = []

for d in tqdm(barrier_chunks):
    v = embedding_model.encode(d['section'])
    barrier_embeddings.append(v)

barrier_embeddings = np.array(barrier_embeddings)

barrier_vindex = VectorSearch()
barrier_vindex.fit(barrier_embeddings, barrier_chunks)

  0%|          | 0/37 [00:00<?, ?it/s]

In [28]:
def text_search(query):
    return barrier_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return barrier_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)

    # Combine and dedupe results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)

    return combined_results

In [33]:
pprint(text_search('How do I install barrier in Mac OS?'))

[{'filename': 'barrier-wiki-master/Troubleshooting.md',
  'section': '## Document Conventions\n'
             '\n'
             '- Text in angle brackets `<text>` is input supplied by the user\n'
             '- Text in _italics_ are user interface elements, items '
             'on-screen, or filenames\n'
             '- Text in a `monospaced font` are keyboard shortcuts, prompts, '
             'commands, or command output'},
 {'filename': 'barrier-wiki-master/Troubleshooting.md',
  'section': '## Things to Check\n'
             '\n'
             '- [The Command Line](#command_line)\n'
             '- [Processes Running](#processes_running)\n'
             '- [Server Reachability](#server_reachability)\n'
             '- [Server Listening](#server_listening)\n'
             '- [Multiple Instances](#multiple_instances)\n'
             '- [Fingerprint Trust](#fingerprint_trust)'},
 {'filename': 'barrier-wiki-master/Troubleshooting.md',
  'section': '## <a name="command_line">Command Li

In [34]:
pprint(vector_search('How do I install barrier in Mac OS?'))

[{'filename': 'barrier-wiki-master/Building-on-macOS.md',
  'section': '## Before You Start\n'
             '\n'
             'You should be at least moderately comfortable with Git, CMake, '
             'and the command\n'
             'line before relying on this method. It requires a good bit of '
             'free hard drive \n'
             'space due to the large amount of dependencies required for the '
             'build (see below)\n'
             'and is meant for programmers that want to help discover and fix '
             'programming \n'
             'errors. When in doubt go for [binaries](Home) instead!\n'
             '\n'
             '# How to Build Barrier.app on MacOS\n'
             '\n'
             '*as of May 2020*\n'
             '\n'
             '- Install Xcode from the App Store or download and install it \n'
             'from the [Apple Developer '
             'Website](https://developer.apple.com/download/more/?=xcode)\n'
             '- Install eit

In [32]:
pprint(hybrid_search('how do I install barrier in Linux?'))

[{'filename': 'barrier-wiki-master/Building-on-Linux.md',
  'section': '## Binaries Built\n'
             '\n'
             '- `barrier` the GUI front-end and "main" program\n'
             '- `barriers` the CLI server component\n'
             '- `barrierc` the CLI client component\n'
             '\n'
             "**note**: these aren't the only things build at compile time, "
             'but merely what the end-user is expected to use.'},
 {'filename': 'barrier-wiki-master/Troubleshooting.md',
  'section': '## <a name="command_line">Command Line</a>\n'
             '\n'
             'Most of the tools used to troubleshoot in this guide are run '
             'from a text-based command line. These are some possible steps to '
             'get to the command line interface.\n'
             '\n'
             '__Note:__ None of the below commands should require elevated '
             'access (`Administrator cmd.exe`, `sudo`, `su`, `doas`, or '
             'equivalent) unless speci